# BrandSight — YOLO retrain v2 (Google Colab)

Retrain after adding video frames to Roboflow.

1. Colab secret: `ROBOFLOW_API_KEY`
2. Set `DATASET_VERSION` (new Roboflow version)
3. Pick `BASE_MODEL` (`yolov8m.pt` or `yolo11m.pt` / `yolo11l.pt`)
4. Run all cells → download `best.pt` → replace `models/best.pt` → redeploy


In [1]:
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# --- Edit these before training ---
ROBOFLOW_WORKSPACE = "s-workspace-jfjba"
ROBOFLOW_PROJECT = "coca-pepsi-juhuf-ndbqf"
DATASET_VERSION = 1  # bump after you generate a new Roboflow version

# Option A (original plan): BASE_MODEL = "yolov8m.pt"
# Option B (higher accuracy): BASE_MODEL = "yolo11m.pt"  # or "yolo11l.pt" if GPU allows
BASE_MODEL = "yolo11l.pt"

EPOCHS = 50
IMGSZ = 1280
BATCH = 8 if IMGSZ >= 1280 else 16
PATIENCE = 20
RUN_NAME = "brandsight_v2"

In [4]:
from pathlib import Path

from google.colab import userdata
from roboflow import Roboflow


rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(DATASET_VERSION).download("yolov11")

DATA_YAML = Path(dataset.location) / "data.yaml"
print("Dataset:", dataset.location)
print("data.yaml:", DATA_YAML)
assert DATA_YAML.exists(), f"Missing {DATA_YAML}"

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Coca-Pepsi-1 in yolov11:: 100%|██████████| 2261/2261 [00:00<00:00, 3254.79it/s]


Dataset: /content/Coca-Pepsi-1
data.yaml: /content/Coca-Pepsi-1/data.yaml


In [5]:
from pathlib import Path

from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    name=RUN_NAME,
    project="brandsight_runs",
)

weights_dir = Path(results.save_dir) / "weights"
best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

print("Run dir:", results.save_dir)
print("best.pt:", best_pt)
print("last.pt:", last_pt)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Coca-Pepsi-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=

In [6]:
import shutil
from pathlib import Path

drive_best = Path("/content/drive/MyDrive/brandsight") / f"{RUN_NAME}_best.pt"
drive_best.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, drive_best)

print("Copied to Google Drive:", drive_best)
print("\nReplace in repo:")
print("  cp <downloaded_best.pt> models/best.pt")

Copied to Google Drive: /content/drive/MyDrive/brandsight/brandsight_v2_best.pt

Replace in repo:
  cp <downloaded_best.pt> models/best.pt


In [7]:
# Optional: quick check on one frame (upload to /content/ first)
# val_model = YOLO(str(best_pt))
# val_model.predict(source="/content/test_frame.jpg", save=True, conf=0.25, imgsz=IMGSZ)